# CS439 Final Project — Telco Churn (Segmentation + Supervised Models)

This notebook mirrors `scripts/reproduce.py` for interactive exploration. Run from the repository root so `src/` resolves.

In [ ]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.preprocess import load_raw, clean_dataframe, stratified_split, fit_scaler, transform
from src.clustering import kmeans_range_inertia, kmeans_range_silhouette, fit_kmeans, pca_2d
from src.train_eval import train_models, evaluate_models, append_cluster_onehot

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
csv_path = ROOT / "data" / "Telco-Customer-Churn.csv"
assert csv_path.exists(), "Run scripts/download_data.py first"
df = load_raw(csv_path)
X, y = clean_dataframe(df)
X_train, X_test, y_train, y_test = stratified_split(X, y)
scaler, X_train_s = fit_scaler(X_train)
X_test_s = transform(scaler, X_test)
X_train_s.shape, X_test_s.shape, y_train.mean(), y_test.mean()

In [ ]:
ks, inertias = kmeans_range_inertia(X_train_s, 2, 10)
_, sils = kmeans_range_silhouette(X_train_s, 2, 10)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ks, inertias, marker="o"); ax[0].set_title("Elbow")
ax[1].plot(ks, sils, marker="o"); ax[1].set_title("Silhouette")
plt.show()

In [ ]:
n_clusters = 4
km = fit_kmeans(X_train_s, n_clusters=n_clusters)
c_train = km.predict(X_train_s)
c_test = km.predict(X_test_s)
Z = pca_2d(np.vstack([X_train_s, X_test_s]))
labels_vis = np.concatenate([c_train, c_test])
plt.figure(figsize=(6, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=labels_vis, cmap="tab10", s=10, alpha=0.65)
plt.colorbar(label="cluster")
plt.title("PCA colored by K-Means cluster")
plt.show()

In [ ]:
models = train_models(X_train_s, y_train.to_numpy())
metrics = evaluate_models(models, X_test_s, y_test.to_numpy())
pd.DataFrame(metrics).T

In [ ]:
X_train_h = append_cluster_onehot(X_train_s, c_train, n_clusters)
X_test_h = append_cluster_onehot(X_test_s, c_test, n_clusters)
models_h = train_models(X_train_h, y_train.to_numpy())
metrics_h = evaluate_models(models_h, X_test_h, y_test.to_numpy())
pd.DataFrame(metrics_h).T